# Lecture 11 – Imputation
### DATA 2201, Fall 2026

### Lecture Outline
- K–S statistic for comparing distributions
- Investigating missingness
- Deletion and mean imputation
- Effects on mean and variability

**Required files:** `data/midparent.csv` and `lec11_utils.py`. Run setup before class; focus on plots and interpretation.

Import the libraries and configure plotting.


In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib_inline.backend_inline import set_matplotlib_formats
from IPython.display import display, IFrame, HTML
from pathlib import Path

import plotly
import plotly.figure_factory as ff
import plotly.graph_objects as go
from scipy.stats import gaussian_kde
import plotly.express as px
from plotly.subplots import make_subplots

pd.options.plotting.backend = "plotly"
plt.rcParams["figure.figsize"] = (10, 5)

Load the lecture’s plotting and summary helpers.


In [13]:
# Import several helper functions for this lecture, 
#  Used for plotting and permutation tests
from lec11_utils import *

## 1. Review: Missingness and permutation tests

- **MCAR:** Missingness does not depend on observed or missing values.
- **MAR:** After conditioning on relevant observed variables, missingness does not depend on the missing values.
- Compare an observed variable between rows with and without a missing value to look for associations.

**Permutation test:**
1. Compute the observed statistic.
2. Shuffle values between groups under the null.
3. Recompute the statistic many times.
4. Compare with the observed statistic to find the p-value.

Use a difference in means for a mean comparison; TVD for categorical distributions. Today we add **K–S** for numerical distribution differences.


## 2. Why the mean is not enough

**Example A:** Two normal distributions with different centers. The difference in means captures this difference.

Run the cell and compare the centers; the plotting code is provided.


Generate two distributions with different means and plot them.


### Helper Function: `create_kde_plotly`

This helper creates KDE plots for lecture demonstrations. You do not need to study its implementation; run the cell and focus on interpreting the plots.

In [14]:
def create_kde_plotly(df, group_col, group_a, group_b, value_col, title):
    fig = go.Figure()

    values = df[value_col].dropna()
    x = np.linspace(values.min() - 1, values.max() + 1, 500)

    for group in [group_a, group_b]:
        sample = df.loc[df[group_col] == group, value_col].dropna()
        kde = gaussian_kde(sample)

        fig.add_trace(go.Scatter(
            x=x,
            y=kde(x),
            mode="lines",
            name=str(group)
        ))

    fig.update_layout(
        title=title,
        xaxis_title=value_col,
        yaxis_title="Density",
        template="plotly_white"
    )
    fig.show()

In [15]:
np.random.seed(42) # So that we get the same results each time (for lecture).

N = 1000 # Number of samples for each distribution.

# Distribution 'A'.
distr1 = pd.Series(np.random.normal(0, 1, size=N // 2))

# Distribution 'B'.
distr2 = pd.Series(np.random.normal(3, 1, size=N // 2))

data = pd.concat([distr1, distr2], axis=1, keys=['A', 'B']).unstack().reset_index().drop('level_1', axis=1)
data = data.rename(columns={'level_0': 'group', 0: 'data'})

meanA, meanB = data.groupby('group')['data'].mean().round(7).tolist()
create_kde_plotly(data, 'group', 'A', 'B', 'data', f'mean of A: {meanA}<br>mean of B: {meanB}')

**Example B:** A mixture of two normal distributions versus one normal distribution with approximately the same mean and standard deviation.

**Ask:** Could two distributions have similar means but different shapes?


Generate distributions with similar means but different shapes.


In [ ]:
np.random.seed(42) # So that we get the same results each time (for lecture).

N = 1000 # Number of samples for each distribution.

# Distribution 'A'.
a = pd.Series(np.random.normal(0, 1, size=N//2))
b = pd.Series(np.random.normal(4, 1, size=N//2))
distr1 = pd.concat([a,b], ignore_index=True)

# Distribution 'B'.
distr2 = pd.Series(np.random.normal(distr1.mean(), distr1.std(), size=N))

data = pd.concat([distr1, distr2], axis=1, keys=['A', 'B']).unstack().reset_index().drop('level_1', axis=1)
data = data.rename(columns={'level_0': 'group', 0: 'data'})

meanA, meanB = data.groupby('group')['data'].mean().round(7).tolist()
create_kde_plotly(data, 'group', 'A', 'B', 'data', f'mean of A: {meanA}<br>mean of B: {meanB}')

## 3. CDFs and the K–S statistic

A **CDF**, $F(x)$, gives the proportion of values less than or equal to $x$.
For example, $F(70)=0.8$ means 80% of values are at most 70.

The **K–S statistic** is the largest vertical gap between two CDFs:

$D=\max_x |F_A(x)-F_B(x)|$.

<center><img src=./imgs/KS2_Example.png width=50%></center>

Larger $D$ means a greater distribution difference. Compare the density plots and CDFs below; 

Plot the distributions beside their CDFs; compare the largest vertical gap.


In [ ]:
fig1 = create_kde_plotly(data, 'group', 'A', 'B', 'data', f'Distributions of A and B')

# Think about what this function is doing!
def create_cdf(group):
    return data.loc[data['group'] == group, 'data'].value_counts(normalize=True).sort_index().cumsum()

fig2 = go.Figure()

fig2.add_trace(
    go.Scatter(x=create_cdf('A').index, y=create_cdf('A'), name='CDF of A')
)

fig2.add_trace(
    go.Scatter(x=create_cdf('B').index, y=create_cdf('B'), name='CDF of B')
)

fig2.update_layout(title='CDFs of A and B')

from plotly.subplots import make_subplots

for i in range(2):
    fig2.data[i]['marker']['color'] = fig1.data[i]['marker']['color']
    fig2.data[i]['showlegend'] = False
    
fig = make_subplots(rows=1, cols=2, subplot_titles=['Distributions', 'CDFs'])
fig.add_trace(fig1.data[0], row=1, col=1)
fig.add_trace(fig1.data[1], row=1, col=1)
fig.add_trace(fig2.data[0], row=1, col=2)
fig.add_trace(fig2.data[1], row=1, col=2)
fig.update_layout(width=1000, height=400);
fig


Compute the observed mean difference and K–S statistic.


In [ ]:
from scipy.stats import ks_2samp

sample_a = data.loc[data['group'] == 'A', 'data']
sample_b = data.loc[data['group'] == 'B', 'data']
observed_diff = abs(sample_a.mean() - sample_b.mean())
observed_ks = ks_2samp(sample_a, sample_b).statistic
print(f"Observed mean difference: {observed_diff:.3f}")
print(f"Observed K–S statistic: {observed_ks:.3f}")


### One shuffle, two statistics

**Null:** A and B come from the same distribution.
**Alternative:** Their distributions differ.

Use the same shuffled samples to compare the mean statistic with K–S. Trace one iteration; the remaining iterations repeat it.


Shuffle values 500 times, compute both statistics, and estimate their p-values.


In [ ]:
n_repetitions = 500
shuffled = data.copy()
diff_means, ks_stats = [], []

for _ in range(n_repetitions):
    shuffled['data'] = np.random.permutation(data['data'])
    groups = shuffled.groupby('group')['data']
    a, b = groups.get_group('A'), groups.get_group('B')
    diff_means.append(abs(a.mean() - b.mean()))
    ks_stats.append(ks_2samp(a, b).statistic)

# Add-one correction avoids reporting a simulated p-value of exactly zero.
p_mean = (1 + np.count_nonzero(np.array(diff_means) >= observed_diff)) / (n_repetitions + 1)
p_ks = (1 + np.count_nonzero(np.array(ks_stats) >= observed_ks)) / (n_repetitions + 1)
print(f"Permutation p-value (mean difference): {p_mean:.3f}")
print(f"Permutation p-value (K–S): {p_ks:.3f}")


Plot the simulated statistics; red lines mark the observed values.


In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=['Mean difference under the null', 'K–S under the null'])
fig.add_trace(go.Histogram(x=diff_means, nbinsx=30, histnorm='probability', showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=ks_stats, nbinsx=30, histnorm='probability', showlegend=False), row=1, col=2)
fig.add_vline(x=observed_diff, line_color='red', row=1, col=1)
fig.add_vline(x=observed_ks, line_color='red', row=1, col=2)
fig.update_layout(height=350, width=1000)
fig


**Interpretation:** For these samples, the mean comparison misses the shape difference, while K–S detects it. Red lines mark the observed statistics.

`ks_2samp(a, b)` also returns a p-value using SciPy's built-in K–S calculation; it does not use the permutation loop above.

**Choose the statistic to match the question:** difference in means for average or directional differences; K–S for general distribution differences.


## 4. Heights: Investigating missingness

Use the same parent–child heights dataset and missingness rule as before: child heights are more likely to be missing for tall fathers or female children.

This is a **known MAR simulation** because missingness is generated using observed father height and gender.


Load the heights data and simulate missing child heights using father height and gender.


In [ ]:
heights_path = Path('data') / 'midparent.csv'
heights = pd.read_csv(heights_path).rename(columns={'childHeight': 'child'})[['father', 'mother', 'gender', 'child']]

def make_missing(r):
    rand = np.random.uniform() # Random real number between 0 and 1.
    if r['father'] > 72 and rand < 0.5:
        return np.nan
    elif r['gender'] == 'female' and rand < 0.3:
        return np.nan
    else:
        return r['child']
    
heights_mar = heights.copy()
heights_mar['child'] = heights_mar.apply(make_missing, axis=1)
heights_mar['child_missing'] = heights_mar['child'].isna()

Plot father heights for rows with and without missing child heights.


In [ ]:
create_kde_plotly(heights_mar[['child_missing', 'father']], 
                  'child_missing', True, False, 'father',
                  "Father's Height by Missingness of Child Height (MAR example)")

Use K–S to compare father-height distributions between the two missingness groups.


In [ ]:
ks_2samp(heights_mar.query('child_missing')['father'], heights_mar.query('not child_missing')['father'])

Compare father heights between rows with and without missing child heights.
A small p-value provides evidence of an association with father height, against MCAR.
**The test alone cannot distinguish MAR from MNAR**; we know MAR here from the simulation rule.


## 5. Handling Missing Values

Missing values are commonly handled using **deletion** or **imputation**.

### Deletion

**Deletion** removes observations with missing values.

- **MCAR:** Deletion generally does not introduce systematic bias because the missing values occur randomly. However, losing observations reduces the sample size and **decreases precision**.
- **MAR or MNAR:** Deletion can introduce **bias** because the missing observations may systematically differ from the observed ones.

**Note on listwise deletion:** If a value needed for the analysis is missing, the entire observation (row) is excluded from that analysis. Missing values in variables not used in the analysis do not require the row to be removed.

### Imputation

**Imputation** replaces missing values with estimated or plausible values instead of removing the observations.

We will use the heights dataset to examine imputation under two missingness mechanisms:

- **MCAR:** Missingness occurs randomly.
- **MAR:** Missingness depends on an observed variable, such as gender.

**Goal:** Compare how imputation affects the distribution and summary statistics of the data under different missingness mechanisms.

Define a helper that randomly removes values to simulate MCAR.


In [ ]:
def make_mcar(data, col, pct=0.5):
    """Create MCAR from complete data"""
    missing = data.copy()
    idx = data.sample(frac=pct, replace=False).index
    missing.loc[idx, col] = np.nan
    return missing

Define a helper that makes missingness depend on a category, such as gender.


In [ ]:
def make_mar_on_cat(data, col, dep_col, pct=0.5):
    """Create MAR from complete data. The dependency is
    created on dep_col, which is assumed to be categorical.
    This is only *one* of many ways to create MAR data.
    For the lecture examples only."""

    missing = data.copy()
    # pick one value to blank out a lot
    high_val = np.random.choice(missing[dep_col].unique())
    weights = missing[dep_col].apply(lambda x: 0.9 if x == high_val else 0.1)
    idx = data.sample(frac=pct, replace=False, weights=weights).index
    missing.loc[idx, col] = np.nan

    return missing

Create MCAR and gender-dependent MAR versions of the heights data.


In [ ]:
np.random.seed(42) # So that we get the same results each time (for lecture).
heights_mcar = make_mcar(heights, 'child', pct=0.5)
heights_mar = make_mar_on_cat(heights, 'child', 'gender', pct=0.5)

Compare child-height means and standard deviations across the three datasets.


In [ ]:
multiple_describe({
    'Original': heights,
    'MCAR': heights_mcar,
    'MAR': heights_mar
})

Compare the observed child-height means and standard deviations with the original data.
MCAR should give similar values; gender-dependent missingness can shift the overall mean because one gender is underrepresented.


## 6. Mean imputation

**Mean imputation** fills missing values with the mean of the observed values.

- Preserves the observed mean.
- Reduces variance by placing missing values at the center.
- Does not repair bias in the observed mean.

### MCAR example

Compare the original, unfilled, and mean-imputed child heights in one table and one plot.


Fill MCAR values with the observed mean and compare summary statistics.


In [ ]:
heights_mcar_mfilled = heights_mcar.copy()
heights_mcar_mfilled['child'] = heights_mcar['child'].fillna(heights_mcar['child'].mean())
df_map = {'Original': heights, 'MCAR, Unfilled': heights_mcar, 'MCAR, Mean Imputed': heights_mcar_mfilled}
multiple_describe(df_map)


Compare the MCAR distributions before and after mean imputation.


In [ ]:
multiple_kdes(df_map)

**Question:** Why does the imputed distribution have a spike at the mean?

The mean stays the same as the observed mean, but the standard deviation falls. Imputed values are not new measurements and should not be treated as independent observations for uncertainty calculations.

### MAR example

Now apply the same method when missingness depends on gender.


Fill MAR values with the overall observed mean and compare summary statistics.


In [ ]:
heights_mar_mfilled = heights_mar.copy()
heights_mar_mfilled['child'] = heights_mar['child'].fillna(heights_mar['child'].mean())
df_map = {'Original': heights, 'MAR, Unfilled': heights_mar, 'MAR, Mean Imputed': heights_mar_mfilled}
multiple_describe(df_map)


Compare the MAR distributions before and after mean imputation.


In [ ]:
multiple_kdes(df_map)

**Interpretation:** The unfilled and mean-imputed datasets have the same overall mean. In this example, that mean is biased high relative to the complete data. Filling with the overall mean retains the bias and reduces variability.


## 7. Wrap-up and questions

- Similar means do not imply similar distributions; K–S compares CDFs.
- Missingness tests reveal observed associations, not proof of MAR.
- Mean imputation preserves the observed mean but reduces variability and may retain bias.

**Check:** Does filling missing values with the mean restore the original distribution? Explain using the MCAR and MAR plots.

The remaining material is homework practice: conditional mean imputation and probabilistic imputation.


# Homework (Practice): 
- Please work with your peers to solve the remaining problems.
- Let me know if you encounter any difficulties in understanding.

With our single mean imputation strategy, the resulting female mean height is biased quite high.

In [ ]:
pd.concat([
    heights.groupby('gender')['child'].mean().rename('Original'),
    heights_mar.groupby('gender')['child'].mean().rename('MAR, Unfilled'),
    heights_mar_mfilled.groupby('gender')['child'].mean().rename('MAR, Mean Imputed')
], axis=1).T

### Within-group (conditional) mean imputation

* **Improvement**: Since MAR data are MCAR within each group, we can perform group-wise mean imputation.
    - In our case, since the missingness of `'child'` is dependent on `'gender'`, we can impute separately for each `'gender'`.
    - For instance, if there is a missing `'child'` height for a `'female'` child, impute their height with the mean observed `'female'` height.

- With this technique, the overall mean remains unbiased, as do the within-group means.

- Like with "single" mean imputation, the variance of the dataset is reduced.

### `transform` returns!

- In MAR data, imputation by the overall mean gives a biased estimate of the mean of each group. 
- To obtain an unbiased estimate of the mean within each group, impute using the mean within each group.
- To perform an operation separately to each gender, we `groupby('gender')` and use the `transform` method.

In [ ]:
def mean_impute(s):
    return s.fillna(s.mean())

heights_mar_cond = heights_mar.groupby('gender')['child'].transform(mean_impute).to_frame()
heights_mar_cond['child'].head()

In [ ]:
df_map['MAR, Conditional Mean Imputed'] = heights_mar_cond
multiple_kdes(df_map)

The <span style='color:rgb(231,41,138)'><b>pink distribution</b></span> does a slightly better job of approximating the <span style='color:rgb(27,158,119)'><b>turquoise distribution</b></span> than the <span style='color:rgb(117,112,179)'><b>purple distribution</b></span>, but not by much.

### Conclusion: Imputation with single values

- Imputing missing data in a column with the mean of the column:
    - faithfully reproduces the mean of the observed dataset,
    - reduces the variance, and
    - biases relationships between the column and other columns if the data are not MCAR.
    
- The same is true with other statistics (e.g. median and mode).

## Probabilistic imputation

### Imputing missing values using distributions

- So far, each missing value in a column has been filled in with a constant value.
    - This creates "spikes" in the imputed distributions.

- **Idea**: We can **probabilistically** impute missing data from a distribution.
    - We can fill in missing data by drawing from the distribution of the **non-missing** data.
    - There are 5 missing values? Pick 5 values from the data that aren't missing.
     - How? Using `np.random.choice` or `.sample`.

- If the data are MCAR, then sample from the entire column of present values. If the data are MAR on some categorical column, then sample from the present values separately for each category.

### Example: Probabilistic imputation in the MAR `heights` dataset

Let's use `transform` to call `prob_impute` separately on each `'gender'`.

In [ ]:
def prob_impute(s):
    s = s.copy()
    
    # Step 1: Find the number of missing child heights for that gender.
    num_null = s.isna().sum()
    
    # Step 2: Sample num_null observed child heights for that gender.
    fill_values = np.random.choice(s.dropna(), num_null)
    
    # Step 3: Fill in missing values and return ser.
    s[s.isna()] = fill_values
    return s

In [ ]:
heights_mar_pfilled = heights_mar.copy()
heights_mar_pfilled['child'] = (
    heights_mar
    .groupby('gender')
    ['child']
    .transform(prob_impute)
)
heights_mar_pfilled['child'].head()

In [ ]:
df_map['MAR, Conditionally Probabilistically Imputed'] = heights_mar_pfilled
multiple_kdes(df_map)

The <b><span style="color:#67a61f">green distribution (conditional probabilistic imputation)</span></b> does the best job of approximating the <b><span style="color:#1c9e76">turqoise distribution (the full dataset with no missing values)</span></b>!

_Remember that the graph above is interactive – you can hide/show lines by clicking them in the legend._

### Observations

- With this technique, the missing values were filled in with observed values in the dataset.

- If a value was never observed in the dataset, it will never be used to fill in a missing value.
    - For instance, if the observed heights were 68, 69, and 69.5 inches, we will never fill a missing value with 68.5 inches even though it's a perfectly reasonable height.

- Solution? Create a histogram (with `np.histogram`) to bin the data, then sample from the histogram.
    - See Lab 5, Question 6.

### Randomness

- Unlike mean imputation, probabilistic imputation is **random** – each time you run the cell in which imputation is performed, the results could be different.

- If we're interested in estimating some population **parameter** given our (incomplete) sample, it's best not to rely on just a single random imputation.

- **Multiple imputation**: Generate multiple imputed datasets and aggregate the results!
    - Similar to bootstrapping.

# See you next lecture!